---
**Name:** Maxim Dokukin  
**SID:** 017025164  
**Course:** ISE 201  
**Instructor:** Prof. S. Gupta  
**Date:** December 5, 2025  
**Dataset:** [Daily Exchange Rates per Euro 1999–2022](https://www.kaggle.com/datasets/lsind18/euro-exchange-daily-rates-19992020/versions/59?resource=download)   

---

**RECOMMENDED TO VIEW IN COLLAB:** https://drive.google.com/file/d/1hi2LbAoX0rTWJJy2yTMUfNbNxhyJq1vq/view?usp=sharing   
Otherwise you might encounter plot rendering glitches
---

# 1. Introduction

## Research Question
I am investigating the behavior of global currencies relative to the Euro (EUR) over the past two decades. Specifically, I aim to answer:
1. **Seasonality:** Do specific currencies exhibit statistically significant seasonal patterns (e.g., "January Effect" or "Window Dressing")?
2. **Latent Factors:** Can the variance in global exchange rates be explained by a few principal components (e.g., a "Global Market Risk" factor)?

## Motivation
Understanding foreign exchange (FX) volatility and correlation is critical for portfolio hedging and risk management. As a financist, understanding whether exchange rate movements are idiosyncratic or systemic (PCA) and whether they are deterministic (Seasonality) allows for better derivatives pricing and risk allocation.

## Related Work
While calendar anomalies like the "January Effect" are widely documented in equity markets, recent literature confirms their existence in foreign exchange. Girardin and Salimi Namin (2019) analyzed major currency pairs from 1971–2017 and documented persistent, statistically significant January and December effects in the EUR/USD pair, attributing them to seasonal equity carry trades. Furthermore, Bassett et al. (2022) at the European Central Bank provide evidence of "Window Dressing" in Euro-denominated markets, where financial institutions systematically contract balance sheets at year-end for regulatory reporting, creating predictable liquidity squeezes that impact exchange rates.

This study builds upon their findings by testing these structural anomalies on a broader basket of 41 currencies over the 1999–2022 period.

# 2. Data

## a. Data Source
The dataset is sourced from Kaggle (lsind18), originally aggregated from the European Central Bank (ECB) Statistical Data Warehouse.
[Daily Exchange Rates per Euro 1999–2022](https://www.kaggle.com/datasets/lsind18/euro-exchange-daily-rates-19992020/versions/59?resource=download)

## b. Data Collection
The data represents daily reference rates determined during the daily **concertation procedure** between central banks, typically fixed at 16:00 CET.

## c. Units of Observation
Each row represents a single trading day (Date) containing the exchange rate of the Euro (EUR) against 41 distinct global currencies.

## d. Variables
* **Index:** Date (Temporal).
* **Continuous:** 41 Exchange rates (e.g., `USD`, `GBP`, `JPY`).
* **Target Transformations:** I will primarily study the **Log Returns** ($r_t$) rather than raw prices to ensure stationarity and additive properties for statistical modeling.

## e. Study Type
Observational study of historical time-series data.

## f. Data Preprocessing
I executed the following transformation pipeline to prepare the data for analysis:

1.  **Normalization:** Standardized column nomenclature (lowercased, stripped of special characters) and reshaped the dataset from wide to long format (`date`, `ccy`, `rate`).
2.  **Coercion:** Converted exchange rates to numeric types, coercing invalid entries to `NaN`. Rows with missing `rate` values were excised.
3.  **Return Construction:** Computed daily log-returns to approximate continuous compounding:
    $$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$
    Resulting non-finite values (`NaN`, `inf`) were removed to prevent computational errors in covariance matrix calculations.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from dateutil.easter import easter
import kagglehub


# download data
path = kagglehub.dataset_download("lsind18/euro-exchange-daily-rates-19992020/versions/59")
path += "/euro-daily-hist_1999_2022.csv"
print("Path to dataset files:", path)

# load data
df = pd.read_csv(path)
df.columns = [
    'date' if x == 'Period\\Unit:' else x.replace('[', '').replace(']', '').strip().lower()
    for x in df.columns
]
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')

# reshape to long
df_long = df.melt(id_vars=['date'], var_name='ccy', value_name='rate')
df_long['rate'] = pd.to_numeric(df_long['rate'], errors='coerce')
df_long = df_long.dropna(subset=['rate'])

Using Colab cache for faster access to the 'euro-exchange-daily-rates-19992020' dataset.
Path to dataset files: /kaggle/input/euro-exchange-daily-rates-19992020/euro-daily-hist_1999_2022.csv


In [ ]:
# calculate the dgaps in data
years = pd.RangeIndex(df_long['date'].dt.year.min(), df_long['date'].dt.year.max() + 1)
holidays = []
for y in years:
    e = easter(int(y))
    holidays.extend([
        pd.Timestamp(y, 1, 1),
        pd.Timestamp(e) - pd.Timedelta(days=2),
        pd.Timestamp(e) + pd.Timedelta(days=1),
        pd.Timestamp(y, 5, 1),
        pd.Timestamp(y, 12, 25),
        pd.Timestamp(y, 12, 26),
    ])
holidays = pd.DatetimeIndex(sorted(set(holidays)))
bday_target = pd.offsets.CustomBusinessDay(holidays=holidays)

def target_gap_count(g):
    if g.empty:
        return 0
    full = pd.date_range(g['date'].min(), g['date'].max(), freq=bday_target)
    return int(pd.Index(full).difference(pd.Index(g['date'])).size)

gaps = (
    df_long.sort_values('date')
    .groupby('ccy')
    .apply(target_gap_count)
    .rename('gap_count')
    .to_frame()
)
print("gap counts:")
print(gaps.sort_index().head(5))

gap counts:
                       gap_count
ccy                             
australian dollar              2
brazilian real                 1
bulgarian lev                  1
canadian dollar                2
chinese yuan renminbi          1


/tmp/ipython-input-2196741000.py:26: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [ ]:
# calculate returns
df_long = df_long.sort_values(['ccy', 'date'])
df_long['ret'] = np.log(df_long['rate']).groupby(df_long['ccy']).diff()

# drop na
nonfinite_mask = ~np.isfinite(df_long['ret'])
print("returns dropped:", int(nonfinite_mask.sum()))
df_ret = df_long[~nonfinite_mask].copy()

returns dropped: 40


# 3. Exploratory Data Analysis


## a. Visualizations
To understand the underlying trends, I visualize the exchange rates of major currencies. This visual inspection helps identify structural breaks (e.g., the 2008 Financial Crisis) and long-term trends (e.g., USD depreciation).

In [ ]:
# Filter for major currencies
majors = ['us dollar', 'uk pound sterling', 'swiss franc']
subset = df_long[df_long['ccy'].isin(majors)]

# Plot using Plotly (consistent with the rest of the notebook)
fig = px.line(
    subset,
    x='date',
    y='rate',
    color='ccy',
    title='EUR per Currency (Major Currencies)',
    labels={'rate': 'Exchange Rate', 'date': 'Date', 'ccy': 'Currency'},
    render_mode='webgl'
)

# Apply clean white template
fig.update_layout(template="plotly_white", height=500)
fig.show()

# this graph has wired glitch that prevents correct displaying

In [ ]:
#split all in the bins for more clear and organized output
intervals = [(0, 1), (1, 3), (3, 10), (10, 50), (50, 100), (100, 500), (500, 10000)]

df = df.copy()
df['date'] = pd.to_datetime(df['date'], errors='coerce')
num = df.drop(columns=['date']).apply(pd.to_numeric, errors='coerce')
mx = num.max(skipna=True)

for lo, hi in intervals:
    keep = mx[(mx > lo) & (mx <= hi)].index.tolist()
    print(f"interval {lo}–{hi}:", keep)
    if not keep:
        continue
    long = pd.concat([df['date'], num[keep]], axis=1).melt(id_vars='date', var_name='ccy', value_name='rate').dropna()
    fig = px.line(
            long,
            x='date',
            y='rate',
            color='ccy',
            title=f'Currencies with {lo} < max ≤ {hi}',
            render_mode='webgl'
        )
    fig.show()

interval 0–1: ['cypriot pound', 'uk pound sterling', 'latvian lats', 'maltese lira']


interval 1–3: ['australian dollar', 'bulgarian lev', 'canadian dollar', 'swiss franc', 'new zealand dollar', 'singapore dollar', 'us dollar']


interval 3–10: ['brazilian real', 'danish krone', 'croatian kuna', 'israeli shekel', 'lithuanian litas', 'malaysian ringgit', 'polish zloty', 'romanian leu']


interval 10–50: ['chinese yuan renminbi', 'czech koruna', 'estonian kroon', 'hong kong dollar', 'mexican peso', 'norwegian krone', 'swedish krona', 'slovak koruna', 'turkish lira', 'south african rand']


interval 50–100: ['indian rupee', 'philippine peso', 'russian rouble', 'thai baht']


interval 100–500: ['greek drachma', 'hungarian forint', 'iceland krona', 'japanese yen', 'slovenian tolar']


interval 500–10000: ['korean won']


## b. Outlier Detection
I utilize Z-scores to identify statistical outliers defined as $|z| > 4$. This high threshold isolates extreme currency devaluations or idiosyncratic shocks distinct from standard market volatility.

In [ ]:
# Outliers via z-scores
g = df_ret.groupby('ccy')['ret']
mu = g.transform('mean')
sd = g.transform('std').replace(0, np.nan)

df_ret['z'] = (df_ret['ret'] - mu) / sd
df_ret['outlier'] = df_ret['z'].abs() > 4

print("Top 5 currencies by outlier count:")
print(df_ret[df_ret['outlier']].groupby('ccy').size().sort_values(ascending=False).head(5))

Top 5 currencies by outlier count:
ccy
indonesian rupiah    55
lithuanian litas     55
croatian kuna        48
russian rouble       48
bulgarian lev        46
dtype: int64


## c. Descriptive Statistics
I analyze the volatility (standard deviation of returns) and mean returns. Identifying the most volatile currencies is crucial for risk-weighting in portfolio construction.

In [ ]:
# Aggregating returns statistics
# Sorting by standard deviation to identify the most volatile currencies
stats_df = (
    df_ret.groupby('ccy')['ret']
    .agg(['count', 'mean', 'std', 'min', 'max'])
    .sort_values('std', ascending=False)
)

print("Summary Statistics (Top 10 Most Volatile):")
print(stats_df.head(10))

Summary Statistics (Top 10 Most Volatile):
                    count      mean       std       min       max
ccy                                                              
iceland krona        3550  0.000165  0.018497 -0.841487  0.296526
turkish lira         5895  0.000635  0.013091 -0.309155  0.519754
brazilian real       5628  0.000218  0.011495 -0.140043  0.140500
south african rand   5895  0.000159  0.009713 -0.084878  0.078128
mexican peso         5895  0.000116  0.008982 -0.078339  0.115358
indonesian rupiah    5895  0.000092  0.008840 -0.099531  0.093772
russian rouble       5895  0.000206  0.008715 -0.107733  0.202118
japanese yen         5895 -0.000004  0.007252 -0.061232  0.053963
korean won           5895 -0.000005  0.007170 -0.081388  0.065006
new zealand dollar   5895 -0.000048  0.006874 -0.036442  0.048133


## d. Visualizing Volatility Clusters
I inspect the time-series behavior of returns to detect **heteroscedasticity** (volatility clustering) and assess the normality of distributions. In FX markets, returns often exhibit "fat tails," necessitating robust risk models.

In [ ]:
# Visualize the Top 10 Most Volatile Currencies
fig_vol = px.bar(
    stats_df.head(10).reset_index(),
    x='ccy',
    y='std',
    title='Top 10 Most Volatile Currencies (Std Dev of Log Returns)',
    labels={'std': 'Standard Deviation (Risk)', 'ccy': 'Currency'},
    color='std',
    color_continuous_scale='Reds'
)
fig_vol.update_layout(xaxis_tickangle=-45)
fig_vol.show()

In [ ]:
major_currencies = ['us dollar', 'uk pound sterling', 'japanese yen', 'swiss franc', 'chinese yuan renminbi']
subset = df_ret[df_ret['ccy'].isin(major_currencies)].copy()

# Line Chart for Time Series Inspection
fig = px.line(
    subset,
    x='date',
    y='ret',
    color='ccy',
    title='Volatility Clustering: Daily Log Returns of Major Currencies vs EUR',
    labels={'ret': 'Log Return', 'ccy': 'Currency'}
)
fig.show()

# Histogram for Distribution Inspection
fig_hist = px.histogram(
    subset,
    x='ret',
    color='ccy',
    nbins=100,
    opacity=0.6,
    marginal='box', # Adds a box plot at the top
    title='Distribution of Returns',
    labels={'ret': 'Log Return'}
)
fig_hist.show()

# 4. Inference and Analysis

To address the research questions, I apply two statistical techniques: **Principal Component Analysis (PCA)** to decompose systemic variance, and **Hypothesis Testing** to validate seasonal anomalies.

## a. Principal Component Analysis (PCA)


**Goal:** Reduce the dimensionality of the 40 currency pairs to identify if a single "Global Factor" drives the majority of FX movements against the Euro.

### 1. Applicability
PCA is highly suitable for this domain because FX markets are strongly interconnected. Dimensionality reduction isolates the **Systemic Signal** (Global Risk) from **Idiosyncratic Noise** (Country-specific events).

### 2. Assumption Checks
* **Linearity:** FX returns generally exhibit linear correlations suitable for PCA.
* **Scaling:** Data is centered and scaled (Standardized) to prevent high-variance currencies from dominating the components artificially.
Ce

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

COLORS = {
    'primary': '#2ca02c',  # Profit/Good (Standard Green)
    'diverging': 'RdYlGn', # Red (Low/Bad) to Green (High/Good)
    'heatmap': 'RdYlGn',   # Red-Yellow-Green for correlations
    'alert': '#d62728'     # Risk/Volatility (Standard Red)
}

def run_pca_analysis(data, title, filter_survivors=False):
    print(f"--- Running PCA: {title} ---")

    # 1. Pivot
    wide = data.pivot(index='date', columns='ccy', values='ret')

    # 2. Filter Logic
    if filter_survivors:
        # Keep currencies present in >90% of rows
        counts = wide.count()
        survivors = counts[counts > 0.9 * len(wide)].index
        wide = wide[survivors]
        print(f"Filtering: Kept {len(survivors)} 'Survivor' currencies.")

    # Drop rows with NaN to ensure complete matrix
    wide_clean = wide.dropna()
    print(f"Matrix Shape: {wide_clean.shape}")
    print(f"Date Range: {wide_clean.index.min().date()} to {wide_clean.index.max().date()}")

    # 3. PCA Execution
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(wide_clean)

    pca = PCA(n_components=10)
    comps = pca.fit_transform(scaled_data)
    var_exp = pca.explained_variance_ratio_

    print(f"PC1 Variance: {var_exp[0]:.2%}")
    print(f"Top 3 Variance: {var_exp[:3].sum():.2%}")

    # 4. Visualization
    # A. Scree Plot
    scree_df = pd.DataFrame({'PC': [f'PC{i+1}' for i in range(10)], 'Var': var_exp})
    fig_scree = px.bar(
        scree_df, x='PC', y='Var',
        title=f'{title}: Scree Plot',
        text_auto='.1%',
        color_discrete_sequence=[COLORS['primary']]
    )
    fig_scree.update_layout(height=350, template="plotly_white")
    fig_scree.show()

    # B. PC1 Time Series
    ts_df = pd.DataFrame({'date': wide_clean.index, 'PC1': comps[:, 0]})
    fig_ts = px.bar(
        ts_df, x='date', y='PC1',
        title=f'{title}: PC1 Factor Intensity',
        color='PC1',
        color_continuous_scale=COLORS['diverging']
    )
    fig_ts.update_traces(marker_line_width=0)
    fig_ts.update_layout(template="plotly_white", height=400, showlegend=False)
    fig_ts.show()

    return pca, var_exp

### PCA Approach A (Crisis Snapshot)

In [ ]:
pca_a, var_a = run_pca_analysis(df_ret, "Approach A (Crisis Snapshot)", filter_survivors=False)

--- Running PCA: Approach A (Crisis Snapshot) ---
Matrix Shape: (115, 40)
Date Range: 2000-07-20 to 2000-12-29
PC1 Variance: 56.54%
Top 3 Variance: 65.03%


### PCA Approach B (Survivor Filter)

In [ ]:
pca_b, var_b = run_pca_analysis(df_ret, "Approach B (Survivor Filter)", filter_survivors=True)

--- Running PCA: Approach B (Survivor Filter) ---
Filtering: Kept 31 'Survivor' currencies.
Matrix Shape: (5496, 31)
Date Range: 2000-07-20 to 2022-01-10
PC1 Variance: 34.79%
Top 3 Variance: 51.20%


### 3. Interpretation and Comparison of Approaches

The dual-PCA approach reveals a critical insight: global currency markets exhibit high systemic risk, but the magnitude is strictly time-dependent.

#### **1. Analysis of Approach A: The "Crisis Snapshot" (Naive Intersection)**
* **Result:** PC1 explains a massive **56.54%** of variance; PC2 explains only **4.56%**.
* **Context:** By maximizing the currency count (40), the data intersection restricted the window to **July–December 2000**.
* **Implication:** This period covers the Dot-com crash and early Euro adoption. The dominance of PC1 indicates **"Crisis Correlation"**: during market stress, diversification fails as all assets move in unison (systemic risk drowns out idiosyncratic factors).

#### **2. Analysis of Approach B: The "Structural Reality" (Survivor Filter)**
* **Result:** PC1 explains **34.79%** of variance; **PC2 rises significantly to 12.02%**.
* **Context:** By filtering for survivors, I analyzed the full **22-year history (2000–2022)**.
* **Implication:**
    1.  **PC1 (The Global Factor):** Over two decades, **~35%** of movements are driven by a single global sentiment factor (Risk On/Off).
    2.  **PC2 (The Structural Shift):** The rise of PC2 (4% to 12%) suggests that over long horizons, structural themes—such as the "Commodity Currency" (AUD, CAD) vs. "Safe Haven" (JPY, CHF) divergence—become statistically significant drivers of returns.

#### **3. Conclusion**
Long-term risk management requires accounting for secondary structural drivers (PC2, PC3) that are invisible during short-term panic correlations.

## b. Hypothesis Testing (Seasonality)


**Goal:** Determine if the Mean Log Return of the USD/EUR pair in **December** is statistically different from the rest of the year. In financial markets, this is often referred to as the "Window Dressing" or "Year-End Repatriation" effect.

### 1. Formulation
* **Null Hypothesis ($H_0$):** $\mu_{Dec} = \mu_{Rest}$ (No difference in mean returns).
* **Alternative Hypothesis ($H_A$):** $\mu_{Dec} \neq \mu_{Rest}$ (Statistically significant difference).
* **Confidence Level:** $\alpha = 0.05$ (95%).

### 2. Assumption Checks
* **Independence:** Daily returns are sufficiently independent (efficient market hypothesis).
* **Normality:** Given $n > 250$ (months), the Central Limit Theorem (CLT) applies.
* **Homogeneity of Variance:** I utilize **Welch’s t-test** (`equal_var=False`) to account for volatility clustering inherent in financial time series.

In [ ]:
import pandas as pd
import plotly.express as px
from scipy import stats

usd_daily = df_ret[df_ret['ccy'] == 'us dollar'].copy()
usd_monthly = (
    usd_daily.set_index('date')['ret']
    .resample('ME') # Resample to Month End
    .sum()
    .reset_index()
)
usd_monthly['month'] = usd_monthly['date'].dt.month
usd_monthly['period'] = usd_monthly['month'].apply(lambda x: 'December' if x == 12 else 'Rest of Year')

dec_monthly = usd_monthly[usd_monthly['period'] == 'December']['ret']
rest_monthly = usd_monthly[usd_monthly['period'] == 'Rest of Year']['ret']

# Test A: Welch's T-Test
t_stat, p_val = stats.ttest_ind(dec_monthly, rest_monthly, equal_var=False)

# Test B: Mann-Whitney U Test
u_stat, p_val_u = stats.mannwhitneyu(dec_monthly, rest_monthly, alternative='two-sided')

print(f"Count (December Months):     {len(dec_monthly)}")
print(f"Count (Rest of Year Months): {len(rest_monthly)}")
print(f"\nMean Monthly Return (Dec):   {dec_monthly.mean():.4%}")
print(f"Mean Monthly Return (Rest):  {rest_monthly.mean():.4%}")
print(f"Difference:                  {dec_monthly.mean() - rest_monthly.mean():.4%}")

print(f"\n--- TEST RESULTS ---")
print(f"1. T-Test P-value:       {p_val:.4f}  [{'SIGNIFICANT' if p_val < 0.05 else 'NOT SIGNIFICANT'}]")
print(f"2. Mann-Whitney P-value: {p_val_u:.4f}  [{'SIGNIFICANT' if p_val_u < 0.05 else 'NOT SIGNIFICANT'}]")

monthly_stats = usd_monthly.groupby('month')['ret'].agg(['mean', 'sem']).reset_index()

fig_season = px.bar(
    monthly_stats,
    x='month',
    y='mean',
    error_y='sem', # Adds confidence interval bars
    title='Figure 1: Average Monthly Log Returns (USD/EUR)',
    labels={'mean': 'Average Log Return', 'month': 'Month'},
    color='mean',
    color_continuous_scale='RdBu' # Preserving the style you liked
)
fig_season.update_xaxes(tickmode='linear', tick0=1, dtick=1)
fig_season.add_vline(x=12, line_dash="dash", line_color="green", annotation_text="Hypothesis Target")
fig_season.show()

fig_box = px.box(
    usd_monthly,
    x='period',
    y='ret',
    color='period',
    title='Figure 2: Monthly Return Distributions (Dec vs Rest)',
    points='all',
    hover_data=['date']
)
fig_box.show()

Count (December Months):     23
Count (Rest of Year Months): 254

Mean Monthly Return (Dec):   1.3297%
Mean Monthly Return (Rest):  -0.1365%
Difference:                  1.4662%

--- TEST RESULTS ---
1. T-Test P-value:       0.0425  [SIGNIFICANT]
2. Mann-Whitney P-value: 0.0438  [SIGNIFICANT]


### 3. Interpretation of Seasonality Results

**Statistical Findings:**
The robust hypothesis testing (aggregated by month) yielded p-values of **0.0425** (T-test) and **0.0438** (Mann-Whitney).
* **Result:** Since $p < 0.05$, we **reject the null hypothesis**.
* **Conclusion:** There is statistically significant evidence of a seasonal anomaly. The Euro consistently outperforms the US Dollar in December.

**Economic Significance:**
The analysis reveals a staggering divergence in mean returns:
* **December Average:** +1.33% (Euro strengthens).
* **Rest of Year Average:** -0.14% (Euro weakens/flat).
* **The "December Premium":** The spread is roughly **147 basis points (1.47%)**.

**Business Implication:**
This suggests a structural "Year-End Effect," likely driven by liquidity constraints or repatriation flows where capital moves back into Euro-denominated assets before books close. A corporate treasurer aware of this seasonality should consider hedging strategies that delay USD purchases until January.

# 5. Conclusion

## Summary of Findings
1.  **Systemic Risk (PCA):** The dual-approach analysis revealed that the FX market is highly interconnected but time-dependent.
    * **Crisis Correlation:** In short, volatile windows (Approach A), a single global factor explains **~56%** of all market moves, indicating that diversification fails during market stress.
    * **Structural Reality:** Over the full 22-year history (Approach B), this global factor explains **~35%** of variance, while a secondary "Structural/Commodity" factor emerges to explain **~12%**. This confirms that while the "Risk On/Risk Off" narrative is dominant, distinct regional or commodity-driven themes are significant long-term drivers.

2.  **Seasonality (December Effect):** There is statistically significant evidence ($p < 0.05$) of a structural anomaly in December returns.
    * The Euro consistently appreciates against the USD in December, with an average monthly return of **+1.33%** versus **-0.14%** for the rest of the year.
    * This **~147 basis point premium** suggests a predictable "Year-End Repatriation" effect that is robust across two decades.

## Limitations
* **Survivorship Bias:** The long-term PCA (Approach B) required dropping 9 currencies. This likely understates volatility by excluding failed or hyper-inflationary economies.
* **Sample Size:** For the seasonality test, aggregating to monthly data reduced the sample size from $N \approx 5000$ to $N \approx 270$. While methodologically robust for independence, it reduces statistical power.
* **Linearity Assumption:** PCA assumes linear correlations and may fail to capture non-linear market regimes.

## Future Work
* Train a machine learning model (e.g., LSTM or XGBoost) to predict the next-month USD/EUR exchange rate, utilizing the latent factors identified in the PCA step as feature inputs.

# 6. References

1. **Dataset:** Kaggle/ECB - [Daily Exchange Rates per Euro 1999–2022](https://www.kaggle.com/datasets/lsind18/euro-exchange-daily-rates-19992020/versions/59?resource=download)
2. **Bassett, R., et al. (2022).** *Window dressing of regulatory metrics: Evidence from repo markets*. European Central Bank Working Paper Series, No. 2771.
3. **Girardin, E., & Salimi Namin, F. (2019).** *The January effect in the foreign exchange market: Evidence for seasonal equity carry trades*. Economic Modelling, 81, 422–439.
4. **Hull, J. C. (2018).** *Options, Futures, and Other Derivatives*. Pearson Education.

# 7. Notes


Code and markdown formatting performed by Gemini   
Plots sometimes go blank for unknown reasons